# channel-list-reverse-build — worked example 3: Verify generator/discriminator channel symmetry from one list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `channel-list-reverse-build`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

When both halves are built from one `hidden_channels` list (discriminator forward, generator reversed), the two channel-pair lists are exact mirrors: each generator pair `(a, b)` corresponds to a discriminator pair `(b, a)`, read in reverse order. Checking this invariant catches off-by-one or wrong-reverse bugs before any training.

## Worked solution

We build both pair lists from one source and assert the mirror invariant.

1. **Discriminator pairs.** Zip the list as-is: for `[3, 32, 64, 128]` we get `[(3,32),(32,64),(64,128)]`.
2. **Generator pairs.** Reverse then zip: `gen_channels = [128, 64, 32, 3]`, pairs `[(128,64),(64,32),(32,3)]`.
3. **The mirror relation.** Reversing the discriminator pair *list* and swapping each tuple should reproduce the generator pairs. `[(c,d) for (d,c) in reversed(disc_pairs)]` -> reverse `[(3,32),(32,64),(64,128)]` to `[(64,128),(32,64),(3,32)]`, then swap each to `[(128,64),(64,32),(32,3)]` — identical to `gen_pairs`.
4. **Why it holds.** Slice-reversing the channel list and pairing is algebraically the same as pairing then reversing-and-swapping; this is just two ways of expressing the same mirror. If the check fails, the generator and discriminator would not have matching shapes at corresponding depths.
5. We compute the mirror independently and print whether it matches, demonstrating the symmetry numerically.

In [ ]:
def channel_pairs(hidden_channels):
    disc_pairs = list(zip(hidden_channels[:-1], hidden_channels[1:]))
    gen_channels = hidden_channels[::-1]
    gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))
    return disc_pairs, gen_pairs

hidden = [3, 32, 64, 128]
disc_pairs, gen_pairs = channel_pairs(hidden)
mirror = [(c, d) for (d, c) in reversed(disc_pairs)]
print('disc_pairs:', disc_pairs)
print('gen_pairs :', gen_pairs)
print('mirror==gen:', mirror == gen_pairs)